# Data preparations
We decided to provide a small notebook with how the Wikidata .jsonl files were created based on a "human"-filtering of the Wikidata dumps (see README.md, Populate new index with Wikidata).

We have certain fields and certain languages we're interested in. Running this for all 11 Mio. Wikidata person entries takes around 45 minutes.

In [ ]:
import json, requests, time, os

# download the wikidata json file if we don't have it already
if not os.path.isfile("../select_wikidata_value_dict.json"):
    r = requests.get("https://polybox.ethz.ch/index.php/s/TYDGgH6s7DERk75/download")
    with open("../select_wikidata_value_dict.json", "wb") as f:
        f.write(r.content)

# load once at start
with open("../select_wikidata_value_dict.json", "r", encoding="utf-8") as f:
    selected_wikidata_values_dict = json.load(f)

session = requests.Session()
session.headers.update({
    #'User-Agent': 'Mozilla/5.0 (compatible; myscript/1.0; +http://example.com)',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:106.0) Gecko/20100101 Firefox/106.0'
})

def get_wikidata_value(wikidata_id: str) -> str:
    if wikidata_id in selected_wikidata_values_dict:
        return selected_wikidata_values_dict[wikidata_id]
    params = {
        "action": "wbgetentities",
        "ids": wikidata_id,
        "languages": "de|en|fr|it|mul",
        "format": "json"
    }
    for attempt in range(3):
        resp = session.get("https://www.wikidata.org/w/api.php", params=params)
        try:
            res = resp.json()
            break
        except ValueError:
            time.sleep(5)
    else:
        print(f"failed to fetch {wikidata_id}")
        selected_wikidata_values_dict[wikidata_id] = None
        return None
    ent = res.get("entities", {}).get(wikidata_id, {})
    labels = ent.get("labels", {})
    for lang in ("de","en","fr","it","mul"):
        if lang in labels:
            val = labels[lang]["value"]
            selected_wikidata_values_dict[wikidata_id] = val
            return val
    selected_wikidata_values_dict[wikidata_id] = None
    return None

In [ ]:
rename_wikidata_fields_dict = {
    "P1038": "relative",
    "P106": "occupation",
    "P1196": "mannerOfDeath",
    "P1449": "nickname",
    "P1477": "birthname",
    "P1559": "nativeName",
    "P19": "placeOfBirth",
    "P20": "placeOfDeath",
    "P21": "gender",
    "P227": "GND_ID",
    "P26": "spouse",
    "P27": "countryOfCitizenship",
    "P31": "instanceOf",
    "P509": "causeOfDeath",
    "P551": "residence",
    "P569": "dateOfBirth",
    "P570": "dateOfDeath",
    "P701": "DODIS_ID",
    "P734": "familyName",
    "P735": "givenName",
    "P7902": "GND_ID_2"
    }

In [ ]:
import orjson
from concurrent.futures import ThreadPoolExecutor
# pre‑compute some helpers
skip_keys = {
    "GND_ID",
    "GND_ID_2",
    "dateOfBirth",
    "dateOfDeath",
    "birthname",
    "nativeName",
    "DODIS_ID",
    "nickname"
}
# aliases and descriptions need this lang order as well
lang_order = ("de","en","fr","it","mul") # TODO order?
rename = rename_wikidata_fields_dict  # already defined earlier

def extract_label(labels: dict) -> str | list | None:
    for lang in lang_order:
        if lang in labels:
            if isinstance(labels[lang], dict):
                return labels[lang]["value"]
            elif isinstance(labels[lang], list):
                return [x["value"] for x in labels[lang]]
    return None

def extract_label_list(labels: list) -> str | None:
    output = []
    for label in labels:
        for lang in lang_order:
            if lang in label:
                output.append(labels[lang]["value"])
                break
    if output != []:
        return output
    return None

def process_person(aux: dict) -> bytes | None:
    out: dict = {}
    out["id"] = aux["id"]  # wikidata id
    for pid, claims in aux["claims"].items():
        if pid not in rename:
            continue
        k = rename[pid]
        for i in claims:
            snak = i.get("mainsnak", {})
            dv = snak.get("datavalue", {}).get("value")
            if dv is None:  # missing datavalue
                continue
            # normalise the wikidata object to a string
            if isinstance(dv, str):
                new_v = dv
            elif isinstance(dv, dict):
                new_v = dv.get("id") or dv.get("time", "")[1:] \
                        or dv.get("text")
            else:
                continue
            if new_v in selected_wikidata_values_dict:
                new_v = selected_wikidata_values_dict[new_v]
            elif k not in skip_keys:
                new_v = get_wikidata_value(new_v)
            out.setdefault(k,[]).append(new_v)
    if not out:
        return None
    if "Mensch" not in out.get("instanceOf"):
        return None
    aliases = extract_label(aux.get("aliases", {}))
    if aliases:
        out["aliases"] = aliases
    descriptions = extract_label(aux.get("descriptions", {}))
    if descriptions:
        out["descriptions"] = descriptions
    label = extract_label(aux.get("labels", {}))
    if label:
        out["labels"] = label
        return orjson.dumps(out)  # bytes
    return None

def aux_func_for_multi(line):
    obj = orjson.loads(line)
    out_line = process_person(obj)
    return out_line
# process the file, writing in batches
batch: list[bytes] = []
flush_threshold = 10000
with open("../humans.json",
          "r", encoding="utf-8") as fin, \
     open("../wikidata_people.jsonl", "a+", encoding="utf-8") as fout:
    read_input = []
    for idx,i in enumerate(fin):
        if len(read_input) < 1000:
            read_input.append(i)
        else:
            with ThreadPoolExecutor(max_workers=10) as executor:
                running_tasks = [executor.submit(aux_func_for_multi,x) for x in read_input]
                for running_task in running_tasks:
                    res = running_task.result()
                    if res is not None:
                        batch.append(res)
                fout.write(b"\n".join(batch).decode("utf-8") + "\n")
                read_input.clear()
                batch.clear()
    
    # final flush
    if read_input:
        with ThreadPoolExecutor(max_workers=6) as executor:
               running_tasks = [executor.submit(aux_func_for_multi,x) for x in read_input]
               for running_task in running_tasks:
                   res = running_task.result()
                   if res is not None:
                       batch.append(res)
               fout.write(b"\n".join(batch).decode("utf-8") + "\n")
               read_input.clear()
               batch.clear()

In [ ]:
# when you finish processing a batch (or at the end):
with open("../select_wikidata_value_dict.json", "w", encoding="utf-8") as f:
    json.dump(selected_wikidata_values_dict, f)